# W4 Day2: 前沿趋势 + P2预习

**日期**: 04/29 周二 | **时长**: 3小时

## 学习目标
1. 理解 **MoE (Mixture of Experts)** 的路由机制和稀疏激活原理
2. 掌握 **长上下文** 处理的挑战与解决方案（FlashAttention, RoPE外推）
3. 了解 **Reasoning** 模型（o1/o3）的思维链与测试时计算
4. 熟悉 **Agent** 趋势：ReAct、工具使用、多智能体、MCP协议
5. 掌握 **Embedding** 基础：文本向量化与相似度计算
6. 学会使用 **FAISS** 进行向量检索

## 面试问题
1. MoE模型如何实现稀疏激活？为什么能提高效率？
2. FlashAttention如何减少HBM访问？
3. o1/o3模型的test-time compute scaling是什么？
4. ReAct框架的思考-行动-观察循环如何工作？
5. 对比不同FAISS索引类型的速度/精度/内存权衡
6. RAG系统的核心pipeline是什么？

## 目录
1. MoE (Mixture of Experts) - 路由机制
2. 长上下文 - O(n²)注意力优化
3. Reasoning (o1/o3) - 思维链推理
4. Agent趋势 - ReAct与工具使用
5. Embedding基础 - 文本向量化
6. FAISS入门 - 向量检索
7. P2预习: RAG概念

## 1. MoE (Mixture of Experts)

### 核心思想
- **稀疏激活**：每次只激活部分expert，而非全部参数
- **门控网络**：学习将token路由到最相关的expert
- **规模效率**：参数量大但计算量小

### 关键组件
1. **Router/Gating Network**: $G(x) = \text{Softmax}(\text{TopK}(h \cdot W_g))$
2. **Top-K Gating**: 只选择K个expert（通常K=2）
3. **Load Balancing Loss**: 防止expert坍缩

### 代表模型
- **Mixtral 8x7B**: 8个expert，每次激活2个
- **DeepSeek-MoE**: 细粒度expert + 共享expert
- **Grok-1**: 314B参数，8个expert

$$y = \sum_{i=1}^{N} G(x)_i \cdot E_i(x)$$

其中 $G(x)_i$ 是门控权重，$E_i(x)$ 是第i个expert的输出

In [ ]:
# MoE Router实现 - Top-K Gating + Load Balancing

import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

class MoERouter:
    """MoE门控网络模拟"""
    
    def __init__(self, num_experts=8, top_k=2, hidden_dim=64):
        self.num_experts = num_experts
        self.top_k = top_k
        # 模拟门控权重矩阵
        self.W_gate = np.random.randn(hidden_dim, num_experts) * 0.1
        
    def forward(self, x):
        """前向传播：计算门控分数"""
        # 计算logits
        logits = x @ self.W_gate  # [batch, num_experts]
        
        # Top-K选择
        top_k_indices = np.argsort(logits, axis=-1)[:, -self.top_k:]
        top_k_logits = np.take_along_axis(logits, top_k_indices, axis=-1)
        
        # Softmax归一化
        top_k_weights = np.exp(top_k_logits) / np.exp(top_k_logits).sum(axis=-1, keepdims=True)
        
        return top_k_indices, top_k_weights, logits
    
    def load_balancing_loss(self, logits):
        """负载均衡损失 - 防止expert坍缩"""
        # 计算每个expert被选中的概率
        probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
        # 平均概率
        avg_probs = probs.mean(axis=0)
        # 理想均匀分布
        uniform = np.ones(self.num_experts) / self.num_experts
        # KL散度作为损失
        loss = np.sum(avg_probs * np.log(avg_probs / uniform + 1e-10))
        return loss

# 演示
np.random.seed(42)
batch_size = 100
hidden_dim = 64

# 创建router
router = MoERouter(num_experts=8, top_k=2, hidden_dim=hidden_dim)

# 模拟输入
x = np.random.randn(batch_size, hidden_dim)

# 前向传播
indices, weights, logits = router.forward(x)

# 统计expert使用情况
expert_counts = np.zeros(8)
for idx in indices.flatten():
    expert_counts[idx] += 1

# 计算负载均衡损失
balance_loss = router.load_balancing_loss(logits)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Expert使用分布
colors = ['#c2553a', '#5a6b4a', '#2d2a26', '#8b7355', '#6b8e8e', '#9b59b6', '#e74c3c', '#3498db']
axes[0].bar(range(8), expert_counts, color=colors)
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('被选中次数')
axes[0].set_title('Expert使用分布 (Top-2 Gating)')
axes[0].axhline(y=batch_size * 2 / 8, color='#2d2a26', linestyle='--', label='理想均匀分布')
axes[0].legend()

# 2. Gate权重分布
sample_logits = logits[:10]  # 取前10个样本
im = axes[1].imshow(sample_logits.T, cmap='YlOrRd', aspect='auto')
axes[1].set_xlabel('样本ID')
axes[1].set_ylabel('Expert ID')
axes[1].set_title('Gate Logits分布')
plt.colorbar(im, ax=axes[1])

# 3. Top-K权重
sample_weights = weights[:10]
x_pos = np.arange(10)
width = 0.35
axes[2].bar(x_pos - width/2, sample_weights[:, 0], width, label='Expert 1权重', color='#c2553a')
axes[2].bar(x_pos + width/2, sample_weights[:, 1], width, label='Expert 2权重', color='#5a6b4a')
axes[2].set_xlabel('样本ID')
axes[2].set_ylabel('权重')
axes[2].set_title('Top-2 Expert权重分配')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\n【MoE路由分析】")
print(f"• 总样本数: {batch_size}")
print(f"• 每个样本激活expert数: {router.top_k}")
print(f"• 总激活次数: {batch_size * router.top_k}")
print(f"• 理想均匀分布下每个expert被选中: {batch_size * router.top_k / 8:.0f}次")
print(f"• 实际分布标准差: {expert_counts.std():.2f}")
print(f"• 负载均衡损失: {balance_loss:.4f}")
print(f"\n【关键洞察】")
print(f"• Top-2 Gating实现稀疏激活，只使用2/8=25%的expert")
print(f"• 负载均衡损失确保expert被均匀利用")
print(f"• Mixtral 8x7B: 47B参数但只激活13B")

## 2. 长上下文

### 核心挑战
- **计算复杂度**: $O(n^2 \cdot d)$，n为序列长度
- **内存占用**: 注意力矩阵 $O(n^2)$ 存储
- **位置编码外推**: 超过训练长度时性能下降

### 解决方案

#### 1. FlashAttention
- **核心思想**: 减少HBM（高带宽内存）访问
- **分块计算**: 将注意力矩阵分块，在SRAM中计算
- **在线Softmax**: 避免存储完整注意力矩阵
- **IO复杂度**: 从 $O(n^2)$ 降到 $O(n^2 d / M)$，M为SRAM大小

#### 2. 稀疏注意力
- **局部窗口**: 每个token只关注邻近token
- **全局token**: 特殊token关注所有位置
- **Longformer/BigBird**: 结合局部+全局+随机注意力

#### 3. RoPE外推
- **位置编码**: $f(x, m) = x e^{im\theta}$
- **NTK-aware插值**: 调整频率基数
- **YaRN**: 动态调整温度和缩放

#### 4. 其他技术
- **Sliding Window Attention**: Mistral使用
- **Ring Attention**: 分布式长序列处理
- **Streaming LLM**: 注意力汇聚token

In [ ]:
# 注意力内存访问模式可视化 - HBM vs SRAM

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def visualize_memory_access():
    """可视化FlashAttention的内存访问模式"""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. 传统注意力 - 需要存储完整注意力矩阵
    ax1 = axes[0, 0]
    n = 8  # 序列长度
    attn_matrix = np.random.rand(n, n)
    np.fill_diagonal(attn_matrix, 0.9)  # 自注意力更强
    
    im1 = ax1.imshow(attn_matrix, cmap='YlOrRd', aspect='equal')
    ax1.set_title('传统注意力: 需要存储完整 n×n 矩阵', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Key位置')
    ax1.set_ylabel('Query位置')
    
    # 添加HBM标注
    ax1.annotate('HBM: O(n²)存储', xy=(n/2, n/2), fontsize=10,
                ha='center', color='#c2553a', fontweight='bold')
    
    # 2. FlashAttention - 分块计算
    ax2 = axes[0, 1]
    block_size = 4
    num_blocks = n // block_size
    
    # 创建分块可视化
    block_matrix = np.zeros((n, n))
    for i in range(num_blocks):
        for j in range(num_blocks):
            # 标记当前计算的块
            block_matrix[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = \
                attn_matrix[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
    
    im2 = ax2.imshow(block_matrix, cmap='YlOrRd', aspect='equal')
    ax2.set_title('FlashAttention: 分块在SRAM中计算', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Key块')
    ax2.set_ylabel('Query块')
    
    # 标注块边界
    for i in range(1, num_blocks):
        ax2.axhline(y=i*block_size-0.5, color='#2d2a26', linewidth=2)
        ax2.axvline(x=i*block_size-0.5, color='#2d2a26', linewidth=2)
    
    ax2.annotate('SRAM: O(n²d/M) IO', xy=(n/2, n/2), fontsize=10,
                ha='center', color='#5a6b4a', fontweight='bold')
    
    # 3. 内存层次结构
    ax3 = axes[1, 0]
    
    # 绘制内存层次
    levels = ['HBM\n(80GB)', 'L2 Cache\n(40MB)', 'SRAM\n(20MB)', 'Register\n(1MB)']
    sizes = [80, 40, 20, 1]
    bandwidth = [2, 6, 19, 100]  # TB/s
    colors = ['#c2553a', '#e74c3c', '#5a6b4a', '#2d2a26']
    
    y_pos = np.arange(len(levels))
    bars = ax3.barh(y_pos, bandwidth, color=colors, height=0.6)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(levels)
    ax3.set_xlabel('带宽 (TB/s)')
    ax3.set_title('GPU内存层次结构', fontsize=12, fontweight='bold')
    
    # 添加数值标注
    for i, (bw, size) in enumerate(zip(bandwidth, sizes)):
        ax3.text(bw + 1, i, f'{bw} TB/s\n{size}MB+', va='center', fontsize=9)
    
    # 4. FlashAttention算法流程
    ax4 = axes[1, 1]
    ax4.set_xlim(0, 10)
    ax4.set_ylim(0, 10)
    ax4.axis('off')
    ax4.set_title('FlashAttention算法流程', fontsize=12, fontweight='bold')
    
    # 绘制流程框
    steps = [
        (5, 9, '将Q,K,V分块', '#c2553a'),
        (5, 7, '在SRAM中计算\nQK^T/√d', '#5a6b4a'),
        (5, 5, '在线Softmax\n更新统计量', '#2d2a26'),
        (5, 3, '累加到输出\nO = O + P·V', '#8b7355'),
        (5, 1, '写回HBM\n最终输出O', '#c2553a')
    ]
    
    for x, y, text, color in steps:
        rect = mpatches.FancyBboxPatch((x-2, y-0.7), 4, 1.4, 
                                       boxstyle="round,pad=0.1", 
                                       facecolor=color, alpha=0.8)
        ax4.add_patch(rect)
        ax4.text(x, y, text, ha='center', va='center', 
                fontsize=9, color='white', fontweight='bold')
    
    # 绘制箭头
    for i in range(len(steps)-1):
        ax4.annotate('', xy=(5, steps[i+1][1]+0.7), 
                    xytext=(5, steps[i][1]-0.7),
                    arrowprops=dict(arrowstyle='->', color='#2d2a26', lw=2))
    
    plt.tight_layout()
    plt.show()

# 运行可视化
visualize_memory_access()

print("\n【FlashAttention关键优势】")
print("• 传统注意力: 需要O(n²) HBM存储完整注意力矩阵")
print("• FlashAttention: 分块计算，只在SRAM中操作")
print("• 内存节省: 从O(n²)降到O(n)，无需存储注意力矩阵")
print("• 速度提升: 减少HBM访问，计算-bound而非memory-bound")
print("• 精确计算: 数学上等价，不是近似")
print("\n【实际效果】")
print("• 训练速度: 提升2-4倍")
print("• 序列长度: 支持更长上下文（32K→128K→1M+）")
print("• 内存占用: 减少5-20倍")

## 3. Reasoning (o1/o3)

### 核心概念

#### 1. Chain-of-Thought (CoT)
- **思维链**: 让模型展示推理过程
- **Few-shot CoT**: 提供推理示例
- **Zero-shot CoT**: "Let's think step by step"

#### 2. Test-Time Compute Scaling
- **核心思想**: 推理时使用更多计算资源
- **o1/o3模型**: 在推理时进行"思考"，消耗更多token
- **性能提升**: 复杂问题上显著提升

#### 3. Process vs Outcome Reward

| 类型 | 描述 | 优点 | 缺点 |
|------|------|------|------|
| **Outcome Reward** | 只奖励最终结果 | 简单，易于定义 | 信用分配困难 |
| **Process Reward** | 奖励每一步推理 | 更精细的反馈 | 需要逐步标注 |
| **ORM vs PRM** | Outcome/Process Reward Model | 各有适用场景 | PRM成本高 |

#### 4. o1/o3的关键技术
- **内部思维链**: 模型在输出前进行隐藏的推理
- **强化学习**: 使用RLHF训练推理能力
- **自我验证**: 检查推理过程的一致性
- **计算预算**: 可配置推理时间

### 数学形式化
$$\text{Performance} = f(\text{Compute}_{\text{train}}, \text{Compute}_{\text{test}})$$

传统模型: $\text{Performance} \propto \log(\text{Compute}_{\text{train}})$

o1/o3: $\text{Performance} \propto \log(\text{Compute}_{\text{train}}) + \log(\text{Compute}_{\text{test}})$

## 4. Agent趋势

### ReAct框架
**Re**asoning + **Act**ing = ReAct

```
思考(Thought): 我需要查找今天的天气
行动(Action): search("北京今天天气")
观察(Observation): 北京今天晴，25°C
思考(Thought): 我已经获得天气信息
回答(Answer): 北京今天天气晴朗，气温25°C
```

### Agent核心组件
1. **规划(Planning)**: 任务分解、子目标设定
2. **记忆(Memory)**: 短期(上下文) + 长期(向量数据库)
3. **工具使用(Tool Use)**: API调用、代码执行
4. **反思(Reflection)**: 自我评估、错误修正

### 多智能体系统
- **角色分工**: 不同agent负责不同任务
- **协作通信**: agent间信息交换
- **代表框架**: AutoGen, CrewAI, LangGraph

### MCP协议 (Model Context Protocol)
- **标准化接口**: 统一agent与工具的通信协议
- **工具描述**: 自动发现和调用工具
- **上下文管理**: 跨会话状态保持

### Agent挑战
- **幻觉问题**: agent可能产生错误行动
- **循环问题**: 陷入无限循环
- **安全性**: 工具调用的风险控制
- **评估困难**: 如何评估agent性能

In [ ]:
# 简单ReAct Agent循环实现 - Think→Act→Observe

import re
import json
from typing import Dict, List, Callable

class Tool:
    """工具基类"""
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    def execute(self, query: str) -> str:
        raise NotImplementedError

class SearchTool(Tool):
    """搜索工具模拟"""
    def __init__(self):
        super().__init__("search", "搜索互联网获取信息")
        self.knowledge = {
            "北京天气": "北京今天晴，25°C，微风",
            "Python最新版本": "Python 3.12.0 发布于2023年10月",
            "Transformer论文": "Attention Is All You Need, 2017, Google Brain"
        }
    
    def execute(self, query: str) -> str:
        for key, value in self.knowledge.items():
            if key in query:
                return value
        return f"搜索'{query}'的结果: 未找到相关信息"

class CalculatorTool(Tool):
    """计算器工具"""
    def __init__(self):
        super().__init__("calculator", "执行数学计算")
    
    def execute(self, expression: str) -> str:
        try:
            # 安全地计算数学表达式
            result = eval(expression)
            return f"计算结果: {expression} = {result}"
        except Exception as e:
            return f"计算错误: {str(e)}"

class ReActAgent:
    """ReAct Agent实现"""
    
    def __init__(self, tools: List[Tool], max_steps: int = 5):
        self.tools = {tool.name: tool for tool in tools}
        self.max_steps = max_steps
        self.trace: List[Dict] = []  # 执行轨迹
        
    def think(self, task: str, history: List[Dict]) -> str:
        """思考阶段 - 决定下一步行动"""
        # 模拟LLM的思考过程
        if not history:
            return f"我需要完成任务: {task}. 让我分析需要哪些信息或操作。"
        
        last_obs = history[-1].get('observation', '')
        if '未找到' in last_obs or '错误' in last_obs:
            return f"之前的尝试没有成功，让我换个方式尝试。"
        elif last_obs:
            return f"我已经获得了一些信息: {last_obs[:50]}... 现在我可以回答问题了。"
        return "让我继续思考..."
    
    def decide_action(self, thought: str, task: str) -> tuple:
        """根据思考决定行动"""
        # 简单的规则匹配（实际中由LLM决定）
        task_lower = task.lower()
        
        if '天气' in task:
            return 'search', f"{task[:4]}天气"
        elif 'python' in task_lower or '版本' in task:
            return 'search', 'Python最新版本'
        elif 'transformer' in task_lower or '论文' in task:
            return 'search', 'Transformer论文'
        elif any(op in task for op in ['+', '-', '*', '/', '计算', '加', '减', '乘', '除']):
            # 提取数学表达式
            numbers = re.findall(r'[\d+\-*/().]', task)
            expr = ''.join(numbers)
            return 'calculator', expr if expr else '2+2'
        else:
            return 'search', task
    
    def act(self, tool_name: str, query: str) -> str:
        """执行行动"""
        if tool_name in self.tools:
            return self.tools[tool_name].execute(query)
        return f"未知工具: {tool_name}"
    
    def run(self, task: str) -> str:
        """运行ReAct循环"""
        print(f"\n{'='*60}")
        print(f"任务: {task}")
        print(f"{'='*60}")
        
        self.trace = []
        
        for step in range(self.max_steps):
            print(f"\n--- Step {step + 1} ---")
            
            # Think
            thought = self.think(task, self.trace)
            print(f"💭 思考: {thought}")
            
            # 检查是否可以给出最终答案
            if self.trace and '已经获得' in thought:
                last_obs = self.trace[-1].get('observation', '')
                answer = f"根据收集到的信息，答案是: {last_obs}"
                print(f"✅ 最终答案: {answer}")
                self.trace.append({'step': step, 'type': 'answer', 'content': answer})
                return answer
            
            # Decide action
            tool_name, query = self.decide_action(thought, task)
            print(f"🎯 决策: 使用 {tool_name} 工具，查询: {query}")
            
            # Act
            observation = self.act(tool_name, query)
            print(f"👀 观察: {observation}")
            
            # 记录轨迹
            self.trace.append({
                'step': step,
                'thought': thought,
                'action': tool_name,
                'query': query,
                'observation': observation
            })
        
        # 达到最大步数
        final_answer = f"经过{self.max_steps}步推理，我收集到以下信息: {self.trace[-1]['observation']}"
        print(f"\n⚠️ 达到最大步数限制")
        print(f"最终答案: {final_answer}")
        return final_answer

# 创建工具
tools = [SearchTool(), CalculatorTool()]

# 创建ReAct Agent
agent = ReActAgent(tools, max_steps=3)

# 测试不同任务
test_tasks = [
    "北京今天天气怎么样？",
    "计算 15 * 23 + 47 的结果",
    "Transformer的原始论文是什么？"
]

results = []
for task in test_tasks:
    result = agent.run(task)
    results.append({'task': task, 'result': result})

print("\n" + "="*60)
print("【ReAct Agent总结】")
print("="*60)
print("\n核心流程:")
print("1. Thought (思考): 分析任务，决定下一步")
print("2. Action (行动): 调用工具获取信息")
print("3. Observation (观察): 获取工具返回的结果")
print("4. 重复直到任务完成或达到步数限制")
print("\n优势:")
print("• 可解释性: 每一步都有明确的思考过程")
print("• 灵活性: 可以根据观察调整策略")
print("• 工具使用: 能够调用外部工具扩展能力")
print("\n挑战:")
print("• 循环风险: 可能陷入无限循环")
print("• 效率问题: 步骤过多时效率下降")
print("• 错误传播: 早期错误可能影响后续推理")

## 5. Embedding基础

### 什么是文本嵌入？
- **定义**: 将文本映射到稠密向量空间
- **目的**: 使语义相似的文本在向量空间中接近
- **维度**: 通常768-4096维

### 主流模型

| 模型 | 维度 | 特点 | 适用场景 |
|------|------|------|----------|
| **BGE** | 768/1024 | 中文优化，开源 | 中文检索 |
| **E5** | 768/1024 | 指令微调 | 多语言 |
| **OpenAI Ada** | 1536 | 商业API | 英文为主 |
| **Jina** | 768 | 多语言，长文本 | 跨语言 |
| **GTE** | 768/1024 | 阿里开源 | 中文场景 |

### 对比训练 (Contrastive Training)

$$\mathcal{L} = -\log \frac{e^{\text{sim}(q, k^+) / \tau}}{e^{\text{sim}(q, k^+) / \tau} + \sum_{i=1}^{N} e^{\text{sim}(q, k_i^-) / \tau}}$$

其中:
- $q$: 查询文本
- $k^+$: 正样本（相关文档）
- $k^-$: 负样本（不相关文档）
- $\tau$: 温度参数
- $\text{sim}$: 余弦相似度

### 相似度度量

#### 1. 余弦相似度 (Cosine Similarity)
$$\text{cosine}(a, b) = \frac{a \cdot b}{||a|| \cdot ||b||} = \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum_{i=1}^{n} a_i^2} \cdot \sqrt{\sum_{i=1}^{n} b_i^2}}$$

#### 2. 欧氏距离 (L2 Distance)
$$\text{L2}(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

#### 3. 点积 (Dot Product)
$$\text{dot}(a, b) = \sum_{i=1}^{n} a_i \cdot b_i$$

### Embedding质量评估
- **检索任务**: Recall@K, NDCG
- **分类任务**: 准确率, F1
- **聚类任务**: 轮廓系数
- **语义相似度**: Spearman相关系数

In [ ]:
# 文本Embedding演示 - 余弦相似度计算
# 如果没有真实模型，使用模拟embedding

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 尝试导入sentence_transformers，如果没有则使用模拟
try:
    from sentence_transformers import SentenceTransformer
    USE_REAL_MODEL = True
    print("✅ 使用真实的Sentence Transformer模型")
except ImportError:
    USE_REAL_MODEL = False
    print("⚠️ 未安装sentence_transformers，使用模拟embedding")

def simulate_embedding(text: str, dim: int = 768) -> np.ndarray:
    """模拟文本embedding（基于文本特征）"""
    # 使用文本的hash作为随机种子，保证相同文本得到相同embedding
    np.random.seed(hash(text) % 2**32)
    
    # 基础随机向量
    embedding = np.random.randn(dim)
    
    # 添加语义特征（简单模拟）
    keywords = {
        '天气': [0, 1, 2],
        '晴': [0, 1, 2],
        '雨': [0, 1, 3],
        '机器学习': [10, 11, 12],
        '深度学习': [10, 11, 13],
        '神经网络': [10, 11, 14],
        'NLP': [20, 21, 22],
        '自然语言': [20, 21, 23],
        '文本': [20, 21, 24],
        '计算机视觉': [30, 31, 32],
        '图像': [30, 31, 33],
        '美食': [40, 41, 42],
        '食物': [40, 41, 43],
        '烹饪': [40, 41, 44]
    }
    
    for keyword, indices in keywords.items():
        if keyword in text:
            for idx in indices:
                embedding[idx] += 2.0  # 增强相关维度
    
    # 归一化
    embedding = embedding / np.linalg.norm(embedding)
    return embedding

# 测试文本集合
texts = [
    "今天天气真好，阳光明媚",
    "明天可能会下雨，记得带伞",
    "机器学习是人工智能的一个分支",
    "深度学习使用神经网络进行训练",
    "自然语言处理是NLP的中文翻译",
    "计算机视觉处理图像和视频",
    "我喜欢吃北京烤鸭",
    "这家餐厅的菜品很美味",
    "天气预报说明天有雨",
    "神经网络由多层感知器发展而来"
]

# 获取embeddings
if USE_REAL_MODEL:
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    embeddings = model.encode(texts)
else:
    embeddings = np.array([simulate_embedding(text) for text in texts])

# 计算相似度矩阵
similarity_matrix = cosine_similarity(embeddings)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 相似度矩阵热力图
ax1 = axes[0]
im = ax1.imshow(similarity_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax1.set_xticks(range(len(texts)))
ax1.set_yticks(range(len(texts)))
ax1.set_xticklabels([t[:6]+'...' for t in texts], rotation=45, ha='right', fontsize=8)
ax1.set_yticklabels([t[:6]+'...' for t in texts], fontsize=8)
ax1.set_title('文本相似度矩阵\n(余弦相似度)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax1)

# 2. t-SNE降维可视化
ax2 = axes[1]
if len(embeddings) > 1:
    # 使用t-SNE降维到2D
    perplexity = min(30, len(embeddings) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # 按类别着色
    categories = ['天气', '天气', 'ML', 'ML', 'NLP', 'CV', '美食', '美食', '天气', 'ML']
    category_colors = {
        '天气': '#c2553a',
        'ML': '#5a6b4a',
        'NLP': '#2d2a26',
        'CV': '#8b7355',
        '美食': '#6b8e8e'
    }
    
    for i, (text, cat) in enumerate(zip(texts, categories)):
        ax2.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], 
                   c=category_colors[cat], s=100, alpha=0.7)
        ax2.annotate(text[:6], (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                    fontsize=7, ha='center', va='bottom')
    
    # 添加图例
    for cat, color in category_colors.items():
        ax2.scatter([], [], c=color, s=100, label=cat)
    ax2.legend(loc='upper right', fontsize=8)

ax2.set_title('t-SNE可视化\n(语义空间分布)', fontsize=12, fontweight='bold')
ax2.set_xlabel('t-SNE维度1')
ax2.set_ylabel('t-SNE维度2')

# 3. Top相似度对
ax3 = axes[2]

# 提取上三角（排除对角线）
pairs = []
for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        pairs.append((i, j, similarity_matrix[i, j]))

# 按相似度排序
pairs.sort(key=lambda x: x[2], reverse=True)
top_pairs = pairs[:8]

# 绘制柱状图
pair_labels = [f"{texts[p[0]][:5]}...\nvs\n{texts[p[1]][:5]}..." for p in top_pairs]
similarities = [p[2] for p in top_pairs]
colors = ['#c2553a' if s > 0.7 else '#5a6b4a' if s > 0.5 else '#2d2a26' for s in similarities]

bars = ax3.barh(range(len(top_pairs)), similarities, color=colors, height=0.6)
ax3.set_yticks(range(len(top_pairs)))
ax3.set_yticklabels(pair_labels, fontsize=7)
ax3.set_xlabel('余弦相似度')
ax3.set_title('Top相似文本对', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 1)

# 添加数值标注
for i, (bar, sim) in enumerate(zip(bars, similarities)):
    ax3.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{sim:.3f}', ha='left', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# 打印分析结果
print("\n【Embedding分析结果】")
print(f"\n文本数量: {len(texts)}")
print(f"Embedding维度: {embeddings.shape[1]}")
print(f"\n最相似的3对文本:")
for i, (idx1, idx2, sim) in enumerate(pairs[:3]):
    print(f"  {i+1}. [{sim:.3f}] '{texts[idx1][:15]}...' <-> '{texts[idx2][:15]}...'")

print(f"\n最不相似的3对文本:")
for i, (idx1, idx2, sim) in enumerate(pairs[-3:]):
    print(f"  {i+1}. [{sim:.3f}] '{texts[idx1][:15]}...' <-> '{texts[idx2][:15]}...'")

print("\n【关键洞察】")
print("• 语义相似的文本（如天气相关）在向量空间中距离更近")
print("• 不同主题的文本（如天气vs美食）距离较远")
print("• Embedding能够捕捉文本的语义信息，而非仅仅词法匹配")
print("• 这是RAG系统的基础：通过embedding相似度检索相关文档")

## 6. FAISS入门

### 什么是FAISS？
- **全称**: Facebook AI Similarity Search
开源向量检索库
- **特点**: 高效、可扩展、支持GPU
- **应用**: 推荐系统、RAG、图像检索

### 索引类型对比

| 索引类型 | 速度 | 精度(Recall) | 内存 | 适用场景 |
|---------|------|-------------|------|----------|
| **Flat (暴力搜索)** | 慢 | 100% | 高 | 小数据集，精确搜索 |
| **IVF (倒排索引)** | 快 | 90-99% | 中 | 中等数据集，平衡速度与精度 |
| **HNSW (图索引)** | 很快 | 95-99% | 高 | 高召回率要求 |
| **PQ (乘积量化)** | 很快 | 85-95% | 低 | 大规模数据，内存受限 |
| **IVF+PQ** | 快 | 90-95% | 低 | 超大规模数据集 |

### 关键概念

#### 1. Flat索引 (IndexFlatL2)
- **原理**: 暴力搜索，计算所有向量距离
- **优点**: 精确，无需训练
- **缺点**: $O(n)$ 复杂度，大数据集慢

#### 2. IVF索引 (IndexIVFFlat)
- **原理**: 先聚类，只搜索相关聚类
- **参数**: `nlist` (聚类数), `nprobe` (搜索聚类数)
- **公式**: $\text{Recall} \propto \frac{nprobe}{nlist}$

#### 3. HNSW索引 (IndexHNSWFlat)
- **原理**: 分层可导航小世界图
- **参数**: `M` (连接数), `efConstruction` (构建参数)
- **优点**: 无需训练，动态添加

#### 4. PQ索引 (IndexPQ)
- **原理**: 向量量化压缩
- **参数**: `m` (子量化器数), `nbits` (每子空间位数)
- **压缩比**: $\frac{d \times 32}{m \times nbits}$

### 速度/精度/内存权衡
```
高精度 + 高内存 → Flat
高精度 + 低内存 → HNSW
中等精度 + 低内存 → IVF+PQ
最快速度 → IVF (低nprobe)
```

In [ ]:
# FAISS基础使用 - 创建索引、添加向量、搜索

import numpy as np
import time

# 尝试导入faiss，如果失败则使用模拟
try:
    import faiss
    USE_FAISS = True
    print("✅ 使用真实的FAISS库")
except ImportError:
    USE_FAISS = False
    print("⚠️ 未安装faiss-cpu，使用模拟实现")

class MockFAISSIndex:
    """模拟FAISS索引（用于演示）"""
    
    def __init__(self, d):
        self.d = d
        self.vectors = None
        self.ntotal = 0
        
    def add(self, x):
        if self.vectors is None:
            self.vectors = x.copy()
        else:
            self.vectors = np.vstack([self.vectors, x])
        self.ntotal = self.vectors.shape[0]
    
    def search(self, x, k):
        # 暴力搜索
        dists = np.linalg.norm(x[:, np.newaxis] - self.vectors, axis=2)
        indices = np.argsort(dists, axis=1)[:, :k]
        distances = np.take_along_axis(dists, indices, axis=1)
        return distances, indices

# 生成模拟数据
np.random.seed(42)
d = 128  # 向量维度
nb = 10000  # 数据库大小
nq = 10  # 查询数量

# 生成随机向量
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

# 为查询生成一些近邻（用于验证）
for i in range(nq):
    xb[i] = xq[i] + np.random.randn(d) * 0.01  # 添加少量噪声

print(f"\n数据集信息:")
print(f"• 向量维度: {d}")
print(f"• 数据库大小: {nb}")
print(f"• 查询数量: {nq}")

# 测试不同的索引类型
if USE_FAISS:
    # 使用真实FAISS
    indices = {
        'Flat': faiss.IndexFlatL2(d),
        'IVF (nlist=100)': None,  # 需要训练
        'HNSW': faiss.IndexHNSWFlat(d, 32),
    }
    
    # 创建IVF索引
    quantizer = faiss.IndexFlatL2(d)
    ivf_index = faiss.IndexIVFFlat(quantizer, d, 100)
    ivf_index.train(xb)
    indices['IVF (nlist=100)'] = ivf_index
else:
    # 使用模拟索引
    indices = {
        'Flat': MockFAISSIndex(d),
    }

# 测试每个索引
k = 4  # 返回最近的4个邻居
results = {}

for name, index in indices.items():
    print(f"\n测试索引: {name}")
    
    # 添加向量
    start_time = time.time()
    index.add(xb)
    add_time = time.time() - start_time
    
    # 搜索
    start_time = time.time()
    D, I = index.search(xq, k)
    search_time = time.time() - start_time
    
    results[name] = {
        'add_time': add_time,
        'search_time': search_time,
        'distances': D,
        'indices': I
    }
    
    print(f"  添加时间: {add_time:.4f}s")
    print(f"  搜索时间: {search_time:.4f}s")
    print(f"  第一个查询的最近邻: {I[0]}")
    print(f"  对应距离: {D[0]}")

# 展示搜索结果
print("\n" + "="*60)
print("搜索结果示例（第一个查询）:")
print("="*60)

for name, res in results.items():
    print(f"\n{name}索引:")
    for i in range(k):
        idx = res['indices'][0][i]
        dist = res['distances'][0][i]
        print(f"  #{i+1}: 索引={idx}, 距离={dist:.4f}")

print("\n【FAISS使用要点】")
print("• Flat索引: 精确搜索，适合小数据集 (<10K)")
print("• IVF索引: 需要训练，通过nprobe平衡速度与精度")
print("• HNSW索引: 无需训练，适合动态添加场景")
print("• 所有索引都支持批量搜索 (batch search)")
print("• 向量必须是float32类型")

In [ ]:
# FAISS索引对比 - Flat vs IVF vs HNSW

import numpy as np
import time
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 尝试导入faiss
try:
    import faiss
    USE_FAISS = True
except ImportError:
    USE_FAISS = False
    print("⚠️ 未安装faiss-cpu，使用模拟数据")

def simulate_index_comparison():
    """模拟不同索引的性能对比"""
    # 模拟数据（基于实际FAISS benchmark）
    data_sizes = [1000, 5000, 10000, 50000, 100000]
    
    # 模拟搜索时间 (ms)
    flat_times = [0.5, 2.5, 5.0, 25.0, 50.0]  # O(n)线性增长
    ivf_times = [0.3, 0.8, 1.2, 3.0, 5.0]  # 近似O(√n)
    hnsw_times = [0.2, 0.5, 0.8, 2.0, 3.5]  # O(log n)
    
    # 模拟Recall@10
    flat_recall = [1.0, 1.0, 1.0, 1.0, 1.0]  # 100%精确
    ivf_recall = [0.95, 0.93, 0.92, 0.90, 0.88]  # 稍低
    hnsw_recall = [0.98, 0.97, 0.96, 0.95, 0.94]  # 较高
    
    # 模拟内存占用 (MB)
    flat_memory = [0.5, 2.5, 5.0, 25.0, 50.0]  # 原始大小
    ivf_memory = [0.6, 3.0, 6.0, 30.0, 60.0]  # 略大（索引结构）
    hnsw_memory = [1.0, 5.0, 10.0, 50.0, 100.0]  # 显著更大（图结构）
    
    return data_sizes, {
        'Flat': {'times': flat_times, 'recall': flat_recall, 'memory': flat_memory},
        'IVF': {'times': ivf_times, 'recall': ivf_recall, 'memory': ivf_memory},
        'HNSW': {'times': hnsw_times, 'recall': hnsw_recall, 'memory': hnsw_memory}
    }

# 获取数据
data_sizes, performance = simulate_index_comparison()

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = {'Flat': '#c2553a', 'IVF': '#5a6b4a', 'HNSW': '#2d2a26'}
markers = {'Flat': 'o', 'IVF': 's', 'HNSW': '^'}

# 1. 搜索时间 vs 数据规模
ax1 = axes[0]
for name, data in performance.items():
    ax1.plot(data_sizes, data['times'], marker=markers[name], 
            color=colors[name], label=name, linewidth=2, markersize=8)

ax1.set_xlabel('数据集大小', fontsize=11)
ax1.set_ylabel('搜索时间 (ms)', fontsize=11)
ax1.set_title('搜索时间 vs 数据规模', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# 2. Recall vs 搜索时间
ax2 = axes[1]
for name, data in performance.items():
    ax2.scatter(data['times'], data['recall'], s=100, 
               color=colors[name], marker=markers[name], label=name)

ax2.set_xlabel('搜索时间 (ms)', fontsize=11)
ax2.set_ylabel('Recall@10', fontsize=11)
ax2.set_title('Recall vs 搜索时间权衡', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_xlim(0, 55)
ax2.set_ylim(0.85, 1.02)
ax2.grid(True, alpha=0.3)

# 添加"理想区域"标注
ax2.annotate('理想区域\n(高速+高Recall)', xy=(3, 0.97), fontsize=9,
            ha='center', color='#5a6b4a', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#5a6b4a', alpha=0.1))

# 3. 内存占用 vs 数据规模
ax3 = axes[2]
for name, data in performance.items():
    ax3.plot(data_sizes, data['memory'], marker=markers[name], 
            color=colors[name], label=name, linewidth=2, markersize=8)

ax3.set_xlabel('数据集大小', fontsize=11)
ax3.set_ylabel('内存占用 (MB)', fontsize=11)
ax3.set_title('内存占用 vs 数据规模', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印对比表格
print("\n" + "="*80)
print("FAISS索引性能对比 (数据规模: 100K向量, 维度: 128)")
print("="*80)
print(f"{'索引类型':<15} {'搜索时间(ms)':<15} {'Recall@10':<15} {'内存(MB)':<15} {'特点':<25}")
print("-" * 80)

index_info = {
    'Flat': ('50.0', '100%', '50', '精确搜索，小数据集'),
    'IVF': ('5.0', '88%', '60', '平衡速度与精度'),
    'HNSW': ('3.5', '94%', '100', '高召回率，动态添加')
}

for name, (time_str, recall, memory, feature) in index_info.items():
    print(f"{name:<15} {time_str:<15} {recall:<15} {memory:<15} {feature:<25}")

print("\n【选择建议】")
print("• 数据量 < 10K: 使用Flat索引")
print("• 数据量 10K-1M: 使用IVF (nprobe=32-128)")
print("• 需要高召回率: 使用HNSW")
print("• 内存受限: 使用IVF+PQ")
print("• 需要动态添加: 使用HNSW")

print("\n【关键参数】")
print("• IVF: nlist=√n, nprobe越大越精确但越慢")
print("• HNSW: M=16-64, efConstruction=100-200")
print("• PQ: m=8-32, nbits=8")

In [ ]:
# HNSW图结构可视化

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import NearestNeighbors

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def visualize_hnsw():
    """可视化HNSW图结构"""
    
    # 生成2D点云（便于可视化）
    np.random.seed(42)
    n_points = 50
    points = np.random.randn(n_points, 2) * 2
    
    # 创建多层图结构
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 定义颜色方案
    colors = {
        'layer0': '#c2553a',  # 底层 - 所有点
        'layer1': '#5a6b4a',  # 中层 - 部分点
        'layer2': '#2d2a26',  # 顶层 - 少量点
        'edge': '#8b7355',
        'query': '#e74c3c',
        'path': '#3498db'
    }
    
    # 1. 底层图 (Layer 0) - 所有点，密集连接
    ax1 = axes[0]
    G0 = nx.Graph()
    for i in range(n_points):
        G0.add_node(i, pos=points[i])
    
    # 使用KNN创建连接
    knn = NearestNeighbors(n_neighbors=5)
    knn.fit(points)
    distances, indices = knn.kneighbors(points)
    
    for i in range(n_points):
        for j in indices[i]:
            if i != j:
                G0.add_edge(i, j)
    
    pos = nx.get_node_attributes(G0, 'pos')
    nx.draw(G0, pos, ax=ax1, node_size=50, node_color=colors['layer0'],
            edge_color=colors['edge'], alpha=0.7, width=0.5)
    ax1.set_title('Layer 0 (底层)\n所有点，密集连接', fontsize=12, fontweight='bold')
    ax1.set_xlabel(f'点数: {n_points}, 平均度: {5}')
    
    # 2. 中层图 (Layer 1) - 部分点
    ax2 = axes[1]
    # 随机选择部分点作为Layer 1
    layer1_indices = np.random.choice(n_points, size=20, replace=False)
    layer1_points = points[layer1_indices]
    
    G1 = nx.Graph()
    for i, idx in enumerate(layer1_indices):
        G1.add_node(idx, pos=points[idx])
    
    # Layer 1的连接（更稀疏）
    knn1 = NearestNeighbors(n_neighbors=3)
    knn1.fit(layer1_points)
    distances1, indices1 = knn1.kneighbors(layer1_points)
    
    for i in range(len(layer1_indices)):
        for j in indices1[i]:
            if i != j:
                G1.add_edge(layer1_indices[i], layer1_indices[j])
    
    # 绘制所有点（Layer 1的点高亮）
    ax2.scatter(points[:, 0], points[:, 1], c=colors['layer0'], 
               s=30, alpha=0.3, label='Layer 0')
    ax2.scatter(layer1_points[:, 0], layer1_points[:, 1], 
               c=colors['layer1'], s=80, alpha=0.8, label='Layer 1')
    
    pos1 = nx.get_node_attributes(G1, 'pos')
    nx.draw(G1, pos1, ax=ax2, node_size=0, edge_color=colors['layer1'],
            alpha=0.6, width=1.5)
    ax2.set_title('Layer 1 (中层)\n部分点，稀疏连接', fontsize=12, fontweight='bold')
    ax2.set_xlabel(f'点数: {len(layer1_indices)}, 平均度: {3}')
    ax2.legend(loc='upper right', fontsize=9)
    
    # 3. 顶层图 (Layer 2) + 搜索路径可视化
    ax3 = axes[2]
    # Layer 2只有少量点
    layer2_indices = np.random.choice(layer1_indices, size=5, replace=False)
    layer2_points = points[layer2_indices]
    
    G2 = nx.Graph()
    for idx in layer2_indices:
        G2.add_node(idx, pos=points[idx])
    
    # Layer 2的连接（最稀疏）
    if len(layer2_indices) > 1:
        for i in range(len(layer2_indices)):
            for j in range(i+1, len(layer2_indices)):
                G2.add_edge(layer2_indices[i], layer2_indices[j])
    
    # 绘制所有层
    ax3.scatter(points[:, 0], points[:, 1], c=colors['layer0'], 
               s=20, alpha=0.2, label='Layer 0')
    ax3.scatter(layer1_points[:, 0], layer1_points[:, 1], 
               c=colors['layer1'], s=50, alpha=0.4, label='Layer 1')
    ax3.scatter(layer2_points[:, 0], layer2_points[:, 1], 
               c=colors['layer2'], s=120, alpha=0.9, label='Layer 2')
    
    pos2 = nx.get_node_attributes(G2, 'pos')
    nx.draw(G2, pos2, ax=ax3, node_size=0, edge_color=colors['layer2'],
            alpha=0.8, width=2.0)
    
    # 模拟搜索路径
    query_point = np.array([1.5, 1.5])
    ax3.scatter(query_point[0], query_point[1], c=colors['query'], 
               s=200, marker='*', zorder=10, label='查询点')
    
    # 搜索路径：从顶层开始，逐层下降
    path_points = [
        layer2_indices[0],  # 从顶层最近的开始
        layer1_indices[5],  # 下降到中层
        10,  # 下降到底层
        15,  # 最终结果
        20   # 最终结果
    ]
    
    path_coords = points[path_points]
    ax3.plot(path_coords[:, 0], path_coords[:, 1], 
            color=colors['path'], linewidth=3, linestyle='--', 
            marker='o', markersize=10, label='搜索路径')
    
    ax3.set_title('HNSW搜索过程\n从顶层到底层的贪心搜索', fontsize=12, fontweight='bold')
    ax3.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# 运行可视化
visualize_hnsw()

# 打印HNSW说明
print("\n" + "="*70)
print("HNSW (Hierarchical Navigable Small World) 图结构")
print("="*70)
print("\n【核心思想】")
print("• 多层图结构：顶层稀疏，底层密集")
print("• 每层都是一个"可导航小世界图"(NSW)
print("• 长距离连接在高层，短距离连接在底层")
print("\n【搜索过程】")
print("1. 从顶层任意点开始")
print("2. 在当前层贪心搜索最近邻")
print("3. 下降到下一层，继续搜索")
print("4. 重复直到Layer 0")
print("5. 返回最终的K个最近邻")
print("\n【关键参数】")
print("• M: 每个点的最大连接数（通常16-64）")
print("• efConstruction: 构建时的搜索范围（100-200）")
print("• efSearch: 查询时的搜索范围（越大越精确）")
print("\n【复杂度】")
print("• 构建: O(n * log(n))")
print("• 搜索: O(log(n))")
print("• 内存: O(n * M) - 比Flat索引大")
print("\n【优势】")
print("• 无需训练，动态添加新点")
print("• 高召回率 (>95%)\n")
print("• 搜索速度快 (log n复杂度)")
print("\n【劣势】")
print("• 内存占用大（需要存储图结构）")
print("• 不支持删除操作（需要重建）")
print("• 构建时间较长")

## 7. P2预习: RAG概念

### 什么是RAG？
**RAG** = **R**etrieval-**A**ugmented **G**eneration（检索增强生成）

### 核心思想
- **问题**: LLM知识过时、幻觉、无法访问私有数据
- **解决**: 检索外部知识，注入到prompt中
- **优势**: 知识可更新、可溯源、减少幻觉

### RAG Pipeline
```
1. Chunk (分块)
   └─ 文档 → 按语义/固定长度切分

2. Embed (向量化)
   └─ 文本块 → Embedding模型 → 向量

3. Index (索引)
   └─ 向量 → FAISS/Milvus → 向量数据库

4. Retrieve (检索)
   └─ 用户问题 → Embedding → Top-K相似文档

5. Generate (生成)
   └─ 问题 + 检索结果 → LLM → 回答
```

### 关键技术点

#### 1. 分块策略 (Chunking)
- **固定长度**: 简单但可能切断语义
- **语义分块**: 按段落/章节切分
- **递归分块**: LangChain常用方法
- **重叠窗口**: 保持上下文连贯

#### 2. 检索策略 (Retrieval)
- **向量检索**: 语义相似度
- **关键词检索**: BM25等传统方法
- **混合检索**: 向量 + 关键词
- **重排序**: Cross-encoder精排

#### 3. 生成策略 (Generation)
- **简单拼接**: 问题 + 文档
- **Map-Reduce**: 分别处理再合并
- **Refine**: 迭代优化答案

### RAG vs Fine-tuning

| 维度 | RAG | Fine-tuning |
|------|-----|-------------|
| **知识更新** | 实时更新 | 需要重新训练 |
| **成本** | 低（只需索引） | 高（训练资源） |
| **可解释性** | 高（可溯源） | 低 |
| **幻觉** | 较少 | 可能更多 |
| **适用场景** | 知识密集型 | 风格/格式调整 |

### 评估指标
- **检索质量**: Recall@K, MRR, NDCG
- **生成质量**: Faithfulness, Relevance, Correctness
- **端到端**: EM (Exact Match), F1

### 常见问题与优化
1. **检索不准**: 优化embedding、使用reranker
2. **上下文太长**: 压缩、摘要、选择性检索
3. **知识冲突**: 来源优先级、时间戳排序
4. **延迟高**: 缓存、预计算、异步检索

## 8. 面试: 前沿趋势 Q&A

### Q1: MoE模型如何实现稀疏激活？为什么能提高效率？

**A**: 
1. **门控网络**: 学习将token路由到最相关的expert
2. **Top-K选择**: 只激活K个expert（通常K=2），而非全部
3. **稀疏计算**: 虽然总参数量大，但每次前向传播只使用部分参数
4. **效率提升**: Mixtral 8x7B有47B参数，但只激活13B，计算量与13B模型相当

**关键公式**: $y = \sum_{i=1}^{N} G(x)_i \cdot E_i(x)$，其中只有Top-K个 $G(x)_i > 0$

### Q2: FlashAttention如何减少HBM访问？

**A**:
1. **分块计算**: 将Q,K,V分成小块，在SRAM中计算
2. **在线Softmax**: 避免存储完整的n×n注意力矩阵
3. **重计算**: 反向传播时重新计算，而非存储
4. **IO复杂度**: 从O(n²)降到O(n²d/M)，M为SRAM大小

**核心**: 计算量不变，但内存访问减少，从memory-bound变为compute-bound

### Q3: o1/o3模型的test-time compute scaling是什么？

**A**:
1. **传统模型**: 性能主要取决于训练时的计算量
2. **o1/o3**: 推理时使用更多计算（更长的思维链）来提升性能
3. **内部推理**: 模型在输出前进行隐藏的"思考"过程
4. **scaling law**: $\text{Performance} \propto \log(\text{Compute}_{\text{train}}) + \log(\text{Compute}_{\text{test}})$

**应用**: 数学推理、代码生成、复杂问题求解

### Q4: ReAct框架的思考-行动-观察循环如何工作？

**A**:
1. **Thought (思考)**: 分析当前状态，决定下一步行动
2. **Action (行动)**: 调用工具（搜索、计算、API等）
3. **Observation (观察)**: 获取工具返回的结果
4. **循环**: 重复直到任务完成或达到步数限制

**优势**: 可解释、灵活、能使用外部工具

### Q5: 对比不同FAISS索引类型的速度/精度/内存权衡

**A**:

| 索引 | 速度 | Recall | 内存 | 适用场景 |
|------|------|--------|------|----------|
| Flat | 慢 | 100% | 高 | 小数据集，精确搜索 |
| IVF | 快 | 90-99% | 中 | 中等数据集 |
| HNSW | 很快 | 95-99% | 高 | 高召回率要求 |
| PQ | 很快 | 85-95% | 低 | 大规模，内存受限 |

### Q6: RAG系统的核心pipeline是什么？

**A**:
1. **Chunk**: 文档分块（固定长度/语义/递归）
2. **Embed**: 文本向量化（BGE/E5/OpenAI Ada）
3. **Index**: 建立向量索引（FAISS/Milvus）
4. **Retrieve**: 检索相关文档（向量/混合检索）
5. **Generate**: LLM生成答案（注入检索结果）

**关键优化**: 分块策略、embedding质量、检索策略、reranking

In [ ]:
# 总结打印

print("\n" + "="*70)
print("W4 Day2: 前沿趋势 + P2预习 - 学习总结")
print("="*70)

sections = [
    ("1. MoE (Mixture of Experts)", [
        "• 稀疏激活: 每次只激活Top-K个expert",
        "• 门控网络: 学习token到expert的路由",
        "• 负载均衡: 防止expert坍缩的loss设计",
        "• 代表模型: Mixtral 8x7B, DeepSeek-MoE"
    ]),
    
    ("2. 长上下文处理", [
        "• FlashAttention: 分块计算，减少HBM访问",
        "• 稀疏注意力: 局部窗口 + 全局token",
        "• RoPE外推: NTK-aware插值，YaRN",
        "• 其他: Sliding Window, Ring Attention, Streaming LLM"
    ]),
    
    ("3. Reasoning (o1/o3)", [
        "• Chain-of-Thought: 展示推理过程",
        "• Test-time compute: 推理时使用更多计算",
        "• Process vs Outcome Reward: 细粒度反馈",
        "• 应用: 数学推理、代码生成、复杂问题"
    ]),
    
    ("4. Agent趋势", [
        "• ReAct: Think → Act → Observe循环",
        "• 工具使用: API调用、代码执行、搜索",
        "• 多智能体: 角色分工、协作通信",
        "• MCP协议: 标准化agent与工具的接口"
    ]),
    
    ("5. Embedding基础", [
        "• 文本向量化: 将文本映射到稠密向量空间",
        "• 主流模型: BGE, E5, GTE, Jina",
        "• 对比训练: InfoNCE loss学习语义相似度",
        "• 相似度度量: 余弦相似度、L2距离、点积"
    ]),
    
    ("6. FAISS入门", [
        "• Flat索引: 暴力搜索，100%召回",
        "• IVF索引: 聚类加速，平衡速度与精度",
        "• HNSW索引: 图结构，高召回率，动态添加",
        "• PQ索引: 向量量化压缩，节省内存"
    ]),
    
    ("7. P2预习: RAG概念", [
        "• Pipeline: Chunk → Embed → Index → Retrieve → Generate",
        "• 分块策略: 固定长度、语义分块、重叠窗口",
        "• 检索策略: 向量检索、混合检索、重排序",
        "• 优势: 知识可更新、可溯源、减少幻觉"
    ])
]

for title, points in sections:
    print(f"\n【{title}】")
    for point in points:
        print(f"  {point}")

print("\n" + "="*70)
print("面试重点:")
print("="*70)
print("\n1. MoE稀疏激活机制与效率提升")
print("2. FlashAttention的内存优化原理")
print("3. o1/o3的test-time compute scaling")
print("4. ReAct框架的Think-Act-Observe循环")
print("5. FAISS索引类型的速度/精度/内存权衡")
print("6. RAG系统的核心pipeline")

print("\n" + "="*70)
print("下一步:")
print("="*70)
print("• 完成P2项目: 构建完整的RAG系统")
print("• 实践FAISS: 在真实数据上测试不同索引")
print("• 深入MoE: 阅读Mixtral和DeepSeek论文")
print("• Agent开发: 使用LangChain/LlamaIndex构建agent")

print("\n" + "="*70)
print("✅ W4 Day2学习完成！")
print("="*70)